# Long dataset

## 1. Long dataset building

In [2]:
import pandas as pd
from pathlib import Path
import warnings

# filter User and Filter Warnings for privacy matters
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
dataset_path = Path('../dataset/filled')

In [4]:
ZONES = [
    'IT_NORD',
    'IT_CNOR',
    'IT_CSUD',
    'IT_SUD',  
    'IT_CALA',
    'IT_SICI',
    'IT_SARD', 
    'IT'
]

In [5]:
def load_df(zone):
    df = pd.read_csv(dataset_path / f'{zone}.csv')
    return df

In [6]:
zone_dfs = []

for zone in ZONES:
    df = load_df(zone)
    if zone == 'IT':
        df = df.rename(columns={
            f'PUN': 'prices',
            f'National Load': 'load',
        })
    else:
        df = df.rename(columns={
            f'PZ_{zone}': 'prices',
            f'Load_{zone}': 'load',
        })
        
    df['zone'] = zone 
    zone_dfs.append(df)

df_zonal_long = pd.concat(
    zone_dfs,
    ignore_index=True,
    sort=False
)

df_zonal_long = df_zonal_long.sort_values(
    ['zone', 'datetime_UTC']
).reset_index(drop=True)

print('Dataset dimensions:', df_zonal_long.shape)
print('\nN° observations per zone:')
print(df_zonal_long['zone'].value_counts())

display(df_zonal_long.head())

Dataset dimensions: (797440, 23)

N° observations per zone:
zone
IT_CALA    100775
IT_CNOR    100775
IT_CSUD    100775
IT_NORD    100775
IT_SARD    100775
IT_SICI    100775
IT_SUD     100775
IT          92015
Name: count, dtype: int64


,datetime_UTC,datetime_local,local_hour,prices,load,Biomass,Fossil Gas,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,...,Waste,Wind Onshore,Fossil Hard coal,Fossil Coal-derived gas,Geothermal,Energy storage,Other renewable,zone,Wind Offshore,Geothermal_source
0,2015-12-31 23:00:00+00:00,2016-01-01 00:00:00+01:00,0,50.41211,23927.233,220.0,5588.0,542.0,0.0,1259.0,...,45.0,203.0,1686.0,241.0,706.0,NaN,NaN,IT,0.0,ENTSO-E
1,2016-01-01 00:00:00+00:00,2016-01-01 01:00:00+01:00,1,44.76435,22517.279,216.0,5034.0,519.0,0.0,1163.0,...,46.0,218.0,1719.0,241.0,706.0,NaN,NaN,IT,0.0,ENTSO-E
2,2016-01-01 01:00:00+00:00,2016-01-01 02:00:00+01:00,2,40.01613,21354.678,217.0,4486.0,530.0,0.0,1089.0,...,44.0,267.0,1728.0,241.0,706.0,NaN,NaN,IT,0.0,ENTSO-E
3,2016-01-01 02:00:00+00:00,2016-01-01 03:00:00+01:00,3,37.92076,20219.874,216.0,4275.0,467.0,0.0,1010.0,...,39.0,227.0,1727.0,241.0,705.0,NaN,NaN,IT,0.0,ENTSO-E
4,2016-01-01 03:00:00+00:00,2016-01-01 04:00:00+01:00,4,35.77287,19229.221,215.0,3926.0,426.0,0.0,988.0,...,44.0,212.0,1658.0,241.0,707.0,NaN,NaN,IT,0.0,ENTSO-E


In [7]:
# unique zone-timestamp
duplicates = df_zonal_long.duplicated(
    subset=['datetime_UTC', 'zone']
).sum()

print('Timestamps-zone duplicates:', duplicates)

# verify available columns
print('\nColumns of the dataset')
print(df_zonal_long.columns.tolist())

# verify missing values for each zone
print('\nObservations per zone')
print(
    df_zonal_long.groupby('zone')['datetime_UTC']
    .agg(['count', 'min', 'max'])
)

Timestamps-zone duplicates: 0

Columns of the dataset
['datetime_UTC', 'datetime_local', 'local_hour', 'prices', 'load', 'Biomass', 'Fossil Gas', 'Fossil Oil', 'Hydro Pumped Storage', 'Hydro Run-of-river and poundage', 'Hydro Water Reservoir', 'Other', 'Solar', 'Waste', 'Wind Onshore', 'Fossil Hard coal', 'Fossil Coal-derived gas', 'Geothermal', 'Energy storage', 'Other renewable', 'zone', 'Wind Offshore', 'Geothermal_source']

Observations per zone
          count                        min                        max
zone                                                                 
IT        92015  2015-12-31 23:00:00+00:00  2026-06-30 21:00:00+00:00
IT_CALA  100775  2014-12-31 23:00:00+00:00  2026-06-30 21:00:00+00:00
IT_CNOR  100775  2014-12-31 23:00:00+00:00  2026-06-30 21:00:00+00:00
IT_CSUD  100775  2014-12-31 23:00:00+00:00  2026-06-30 21:00:00+00:00
IT_NORD  100775  2014-12-31 23:00:00+00:00  2026-06-30 21:00:00+00:00
IT_SARD  100775  2014-12-31 23:00:00+00:00  2026-06-30 2

## 2. Load anomaly investigation

In [8]:
# save in CSV format
df_zonal_long.to_csv(dataset_path / 'long-zonal.csv', index=False)

In [9]:
df_zonal_long['load'].loc[df_zonal_long['zone']=='IT_SARD']

495115     997.138
495116     951.110
495117     903.001
495118     855.196
495119     855.747
            ...   
595885    1422.800
595886    1450.400
595887    1419.200
595888    1346.600
595889    1250.500
Name: load, Length: 100775, dtype: float64

In [10]:
df_2 = pd.read_csv(
    Path('../dataset/filled')
    / 'IT_SARD.csv'
)

In [11]:
df_2['Load_IT_SARD']

0          997.138
1          951.110
2          903.001
3          855.196
4          855.747
            ...   
100770    1422.800
100771    1450.400
100772    1419.200
100773    1346.600
100774    1250.500
Name: Load_IT_SARD, Length: 100775, dtype: float64

In [12]:
df_3 = pd.read_excel(
    Path('../dataset/raw/gme/load')
    / 'GME-load-2015.xlsx',
    engine='openpyxl'
)

df_4 = pd.read_excel(
    Path('../dataset/raw/gme/load')
    / 'GME-load-2026.xlsx',
    engine='openpyxl',
)

In [13]:
df_3['Sardegna'].head()

0    997,138
1    951,110
2    903,001
3    855,196
4    855,747
Name: Sardegna, dtype: str

In [14]:
df_4['Sardegna'].tail()

4338    1.422,8
4339    1.450,4
4340    1.419,2
4341    1.346,6
4342    1.250,5
Name: Sardegna, dtype: str

In [15]:
df_5 = pd.read_csv(
    Path('../dataset/clean/load')
    / 'load-clean.csv'
)
df_5['Sardegna']

0          997.138
1          951.110
2          903.001
3          855.196
4          855.747
            ...   
100770    1422.800
100771    1450.400
100772    1419.200
100773    1346.600
100774    1250.500
Name: Sardegna, Length: 100775, dtype: float64